# Hitri preizkus: ali metabolni geni napovejo ekspanzijo klonov z eno celico

Preverba klonov je pokazala, da skupni AUC modela TRIM večinoma meri velikost klona pred zdravljenjem, signal onkraj nje pa je pri klonih z eno celico in samo v različicah z RNA. Ta zvezek brez modela TRIM in brez GPU preveri, iz katerih genov ta signal pride.

**Postopek**

1. Celice krvi pred zdravljenjem pri vseh bolnikih, ki imajo tudi vzorec krvi po zdravljenju. Oznaka je ista kot pri modelu TRIM: klon ima v krvi po zdravljenju več celic kot pred njim.
2. Glavna množica: celice iz klonov z eno celico pred zdravljenjem (velikost klona je pri vseh enaka, zato ne more pojasniti napovedi).
3. Značilke (vsak nabor skrčimo na 30 glavnih komponent): vsi geni, metabolni geni (Human-GEM 1.19.0), ostali geni, 10 naključnih naborov genov enake velikosti in podobnega izražanja kot metabolni, vhod RNA modela TRIM (`data_rna_pca.pkl`), fluksi scFEA (povprečje treh zagonov), TCR, CD4/CD8 in kakovost celice (število UMI in genov).
4. Logistična regresija, za vsakega bolnika naučena na ostalih bolnikih. AUC štejemo samo v parih celic istega bolnika, interval je bootstrap po bolnikih.

**Kako poženeš:** Runtime → Run all. Zadostuje CPU (okoli 10 minut, večino časa vzame nalaganje `data_rna_counts.pkl`). Rezultati in značilke gredo v novo mapo `MyDrive/Diploma/diploma_statistika/preizkus_genov/<čas>/`, obstoječih datotek zvezek ne spreminja. Značilke so shranjene, da lahko analizo nadaljujeva brez ponovnega zagona.

In [ ]:
import os, io, sys, json, time, pickle, hashlib, urllib.request
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # lokalni zagon: poti iz spremenljivk okolja

try:
    from IPython.display import display
except ImportError:
    display = print

PROC_DIR  = os.environ.get('PREIZKUS_PROC', '/content/drive/MyDrive/Diploma/data/processed')
TCR_PKL   = os.path.join(PROC_DIR, 'tcr_ae', 'step_50100', 'data_tcr.pkl')
FLUX_PKL  = [os.path.join(PROC_DIR, 'flux_runs', f'data_flux_seed{s}.pkl') for s in range(3)]
OUT_DIR   = os.path.join(os.environ.get('PREIZKUS_IZHOD', '/content/drive/MyDrive/Diploma/diploma_statistika/preizkus_genov'),
                         time.strftime('%Y%m%d-%H%M%S'))
HUMANGEM_URL = 'https://raw.githubusercontent.com/SysBioChalmers/Human-GEM/v1.19.0/model/genes.tsv'
HUMANGEM_TSV = os.environ.get('PREIZKUS_HUMANGEM', '')   # lokalna kopija genes.tsv (neobvezno)
CSGEM_GENI   = os.environ.get('PREIZKUS_CSGEM', '')      # seznam genov csGEM, en simbol na vrstico (ko bo narejen)

K_PCA         = 30     # glavnih komponent na nabor
N_NAKLJUCNIH  = 10     # naključni nabori genov
MIN_CELIC_GEN = 10     # gen mora biti zaznan v vsaj toliko izbranih celicah
C_LOGREG      = 0.1
N_BOOT        = 2000
SEME          = 0

os.makedirs(OUT_DIR, exist_ok=False)
print('pandas', pd.__version__, '| numpy', np.__version__)
print('Podatki:', PROC_DIR)
print('Izhod:', OUT_DIR)

## 1. Izbor celic in oznaka

In [ ]:
with open(os.path.join(PROC_DIR, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)
with open(os.path.join(PROC_DIR, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)
lab = data_labels.reset_index(drop=True)
STEVILA = df_all_tcrs.fillna(0).values   # kri pred, kri po, tumor pred, tumor po (vsi bolniki skupaj)

kri = lab['Tissue'] == 0
bolniki_po = set(lab.loc[kri & (lab['Treatment Stage'] == 1), 'Patient'])
izbor = kri & (lab['Treatment Stage'] == 0) & lab['Patient'].isin(bolniki_po) & (lab['CDR3(Beta1)'] != -1)
idx = np.flatnonzero(izbor.to_numpy())
tcr = lab['CDR3(Beta1)'].to_numpy()[idx].astype(int)
rc = STEVILA[tcr]

celice = pd.DataFrame({'vrstica': idx, 'bolnik': lab['Patient'].to_numpy()[idx].astype(int), 'tcr': tcr,
                       'cd8': lab['SubCellType'].to_numpy()[idx].astype(int),
                       'krv_pred': rc[:, 0].astype(int), 'krv_po': rc[:, 1].astype(int)})
st_b = lab[kri].groupby(['Patient', 'CDR3(Beta1)', 'Treatment Stage']).size().unstack(fill_value=0)
kljuc = pd.MultiIndex.from_arrays([celice['bolnik'], celice['tcr']])
for faza, ime in [(0, 'krv_pred_bolnik'), (1, 'krv_po_bolnik')]:
    celice[ime] = st_b[faza].reindex(kljuc).fillna(0).to_numpy().astype(int) if faza in st_b.columns else 0
celice['eksp'] = (celice['krv_pred'] < celice['krv_po']).astype(int)
celice['ena_celica'] = celice['krv_pred'] == 1

sing = celice['ena_celica'].to_numpy()
pregled = celice[sing].groupby('bolnik').agg(celic=('eksp', 'size'), eksp=('eksp', 'sum'))
print(f'Izbranih celic: {len(celice)} pri {celice["bolnik"].nunique()} bolnikih; iz klonov z eno celico: {sing.sum()}, '
      f'od tega ekspandiranih {int(celice.loc[sing, "eksp"].sum())} pri {int((pregled["eksp"] > 0).sum())} bolnikih')
display(pregled.T)

## 2. RNA in nabori genov

In [ ]:
with open(os.path.join(PROC_DIR, 'data_rna_counts.pkl'), 'rb') as f:
    counts = pickle.load(f)
with open(os.path.join(PROC_DIR, 'gene_names.pkl'), 'rb') as f:
    geni = np.array([str(g) for g in pickle.load(f)])
assert counts.shape == (len(lab), len(geni)), f'counts {counts.shape} se ne ujema z data_labels {len(lab)} in geni {len(geni)}'
X = sparse.csr_matrix(counts[idx], dtype=np.float32)
del counts

n_umi = np.asarray(X.sum(1)).ravel()
celice['n_umi'] = n_umi.astype(int)
celice['n_genov'] = np.diff(X.indptr)
X = sparse.csr_matrix(sparse.diags(1e4 / n_umi) @ X)   # normalizacija na 10 000 UMI
X.data = np.log1p(X.data)
zaznan = X.getnnz(axis=0) >= MIN_CELIC_GEN

if HUMANGEM_TSV:
    hg = pd.read_csv(HUMANGEM_TSV, sep='\t', dtype=str)
else:
    surovo = urllib.request.urlopen(HUMANGEM_URL, timeout=120).read()
    print('Human-GEM 1.19.0 genes.tsv, sha256', hashlib.sha256(surovo).hexdigest())
    hg = pd.read_csv(io.BytesIO(surovo), sep='\t', dtype=str)
prim, nasi = set(hg['geneSymbols'].dropna()), set(geni)
metabolni_simboli = prim & nasi
for s, sin in zip(hg['geneSymbols'], hg['geneAliases']):   # stara imena v naših podatkih (npr. AARS za AARS1)
    if pd.isna(s) or s in nasi or pd.isna(sin):
        continue
    zad = [a.strip() for a in str(sin).split(';') if a.strip() in nasi and a.strip() not in prim]
    if len(zad) == 1:
        metabolni_simboli.add(zad[0])

vsi = np.flatnonzero(zaznan)
je_met = np.isin(geni[vsi], list(metabolni_simboli))
met, nemet = vsi[je_met], vsi[~je_met]
nabori = {'vsi_geni': vsi, 'metabolni': met, 'nemetabolni': nemet}

# naključni nabori: enako število genov iz vsakega razreda povprečnega izražanja kot pri metabolnih (izbrani izmed nemetabolnih)
povp = np.asarray(X[:, vsi].mean(0)).ravel()
razred = pd.qcut(povp, 20, labels=False, duplicates='drop')
razred_met, razred_nemet = razred[je_met], razred[~je_met]
rng = np.random.default_rng(SEME)
for r in range(N_NAKLJUCNIH):
    izb = []
    for k in np.unique(razred_met):
        kand = nemet[razred_nemet == k]
        izb.append(rng.choice(kand, size=min((razred_met == k).sum(), len(kand)), replace=False))
    nabori[f'nakljucni_{r}'] = np.sort(np.concatenate(izb))

if CSGEM_GENI:
    cs = set(pd.read_csv(CSGEM_GENI, header=None)[0].astype(str))
    nabori['csgem'] = vsi[np.isin(geni[vsi], list(cs))]

for ime, st in nabori.items():
    print(f'  {ime:>13}: {len(st)} genov')
print(f'Human-GEM: {len(prim)} genov, v naših podatkih {len(metabolni_simboli)} (s sinonimi), med zaznanimi geni {len(met)}')

## 3. Značilke

Vsak nabor genov: z-vrednosti po genih (omejene na ±10) in 30 glavnih komponent na vseh izbranih celicah. PCA ne vidi oznak.

In [ ]:
def pca(Z, k=K_PCA):
    Z = np.asarray(Z, dtype=np.float32)
    Z = Z - Z.mean(0)
    sd = Z.std(0); sd[sd == 0] = 1
    Z /= sd
    np.clip(Z, -10, 10, out=Z)
    k = min(k, Z.shape[1] - 1, Z.shape[0] - 1)
    return PCA(n_components=k, svd_solver='randomized', random_state=SEME).fit_transform(Z).astype(np.float32)


t0 = time.time()
znacilke = {}
for ime, st in nabori.items():
    znacilke[ime] = pca(X[:, st].toarray())
print(f'PCA genov: {time.time() - t0:.0f} s')

with open(os.path.join(PROC_DIR, 'data_rna_pca.pkl'), 'rb') as f:
    znacilke['trim_rna_pca'] = np.asarray(pickle.load(f), dtype=np.float32)[idx][:, :K_PCA]

fluksi = []
for pot in FLUX_PKL:
    if os.path.exists(pot):
        with open(pot, 'rb') as f:
            fluksi.append(np.asarray(pickle.load(f), dtype=np.float32)[idx])
print(f'Zagonov scFEA: {len(fluksi)}')
if fluksi:
    znacilke['fluksi'] = pca(np.mean(fluksi, 0))

with open(TCR_PKL, 'rb') as f:
    znacilke['tcr'] = pca(np.asarray(pickle.load(f), dtype=np.float32)[idx])
znacilke['cd8'] = celice[['cd8']].to_numpy(np.float32)
znacilke['kakovost'] = np.column_stack([np.log(celice['n_umi']), np.log(celice['n_genov'])]).astype(np.float32)

for ime, Z in znacilke.items():
    np.save(os.path.join(OUT_DIR, f'X_{ime}.npy'), Z)
for ime, st in nabori.items():
    pd.Series(geni[st]).to_csv(os.path.join(OUT_DIR, f'geni_{ime}.txt'), index=False, header=False)
celice.to_csv(os.path.join(OUT_DIR, 'celice.csv'), index=False)
print('Značilke shranjene:', {k: v.shape for k, v in znacilke.items()})

## 4. Napoved z izpuščanjem po enega bolnika

AUC po bolnikih: primerjamo samo ekspandirane in neekspandirane celice istega bolnika in pare seštejemo čez bolnike. Tako razlike med bolniki (globina vzorčenja, delež ekspandiranih) ne vplivajo na rezultat.

In [ ]:
y = celice['eksp'].to_numpy()
b = celice['bolnik'].to_numpy()


def lopo(Z, maska):
    """Ocena za celice v maski; za vsakega bolnika model, naučen na celicah v maski pri ostalih bolnikih."""
    s = np.full(len(y), np.nan)
    for p in np.unique(b[maska]):
        test, ucna = maska & (b == p), maska & (b != p)
        if y[ucna].sum() == 0:
            continue
        sc = StandardScaler().fit(Z[ucna])
        m = LogisticRegression(C=C_LOGREG, class_weight='balanced', max_iter=5000).fit(sc.transform(Z[ucna]), y[ucna])
        s[test] = m.decision_function(sc.transform(Z[test]))
    return s


def po_bolnikih(s, maska):
    """Za vsakega bolnika z obema razredoma: U (Mann-Whitney) in število parov."""
    vrst = []
    for p in np.unique(b[maska]):
        m = maska & (b == p)
        yp = y[m].astype(bool)
        n1, n0 = int(yp.sum()), int((~yp).sum())
        if n1 == 0 or n0 == 0 or np.isnan(s[m]).any():
            continue
        r = rankdata(s[m])
        vrst.append((p, r[yp].sum() - n1 * (n1 + 1) / 2, n1 * n0, n1, n0))
    return pd.DataFrame(vrst, columns=['bolnik', 'u', 'pari', 'n_eksp', 'n_neeksp'])


ocene = {ime: lopo(Z, sing) for ime, Z in znacilke.items()}
tabele = {ime: po_bolnikih(s, sing) for ime, s in ocene.items()}
bolniki_eval = tabele['vsi_geni']['bolnik'].to_numpy()
for ime, t in tabele.items():
    assert np.array_equal(t['bolnik'].to_numpy(), bolniki_eval)
U = np.column_stack([tabele[i]['u'].to_numpy() for i in ocene])
W = tabele['vsi_geni']['pari'].to_numpy()
imena = list(ocene)

rng = np.random.default_rng(SEME)
C = rng.multinomial(len(W), np.full(len(W), 1 / len(W)), size=N_BOOT)   # bootstrap po bolnikih
auc = U.sum(0) / W.sum()
B = (C @ U) / (C @ W)[:, None]

rez = pd.DataFrame({'znacilke': imena, 'auc': auc,
                    'boot_sp': np.percentile(B, 2.5, axis=0), 'boot_zg': np.percentile(B, 97.5, axis=0)})
rez['n_bolnikov'] = len(W)
rez['n_eksp'] = int(tabele['vsi_geni']['n_eksp'].sum())
rez['n_celic'] = int(tabele['vsi_geni'][['n_eksp', 'n_neeksp']].sum().sum())
display(rez.round(3))

**Primerjave** (razlika AUC, bootstrap po bolnikih). `nakljucni_povp` je povprečje desetih naključnih naborov.

In [ ]:
I = {ime: i for i, ime in enumerate(imena)}
nak = [I[i] for i in imena if i.startswith('nakljucni_')]


def razlika(a_auc, a_boot, b_auc, b_boot, ime_a, ime_b):
    d, db = a_auc - b_auc, a_boot - b_boot
    return {'a': ime_a, 'b': ime_b, 'auc_a': a_auc, 'auc_b': b_auc, 'razlika': d,
            'boot_sp': np.percentile(db, 2.5), 'boot_zg': np.percentile(db, 97.5),
            'boot_p': min(1.0, 2 * min((db <= 0).mean(), (db >= 0).mean()))}


prim = []
nak_auc, nak_boot = auc[nak].mean(), B[:, nak].mean(1)
for a in ['metabolni', 'csgem']:
    if a in I:
        prim.append(razlika(auc[I[a]], B[:, I[a]], nak_auc, nak_boot, a, 'nakljucni_povp'))
        prim.append(razlika(auc[I[a]], B[:, I[a]], auc[I['nemetabolni']], B[:, I['nemetabolni']], a, 'nemetabolni'))
        prim.append(razlika(auc[I[a]], B[:, I[a]], auc[I['vsi_geni']], B[:, I['vsi_geni']], a, 'vsi_geni'))
for a, c in [('vsi_geni', 'cd8'), ('vsi_geni', 'kakovost'), ('vsi_geni', 'fluksi'), ('fluksi', 'cd8'),
             ('vsi_geni', 'trim_rna_pca'), ('tcr', 'cd8')]:
    if a in I and c in I:
        prim.append(razlika(auc[I[a]], B[:, I[a]], auc[I[c]], B[:, I[c]], a, c))
df_prim = pd.DataFrame(prim)
display(df_prim.round(3))
if 'metabolni' in I:
    print(f'Metabolni nabor je boljši od {int((auc[I["metabolni"]] > auc[nak]).sum())} od {len(nak)} naključnih naborov '
          f'(naključni: {auc[nak].min():.3f}–{auc[nak].max():.3f}).')

## 5. Shranjevanje

In [ ]:
po_b = pd.concat([t.assign(znacilke=ime, auc=t['u'] / t['pari']) for ime, t in tabele.items()], ignore_index=True)
rez.to_csv(os.path.join(OUT_DIR, 'rezultati_auc.csv'), index=False)
df_prim.to_csv(os.path.join(OUT_DIR, 'primerjave.csv'), index=False)
po_b.to_csv(os.path.join(OUT_DIR, 'auc_po_bolnikih.csv'), index=False)
pd.DataFrame({ime: s for ime, s in ocene.items()}).assign(vrstica=celice['vrstica']).to_csv(
    os.path.join(OUT_DIR, 'ocene_lopo.csv'), index=False)
with open(os.path.join(OUT_DIR, 'nastavitve.json'), 'w', encoding='utf-8') as f:
    json.dump({'K_PCA': K_PCA, 'N_NAKLJUCNIH': N_NAKLJUCNIH, 'MIN_CELIC_GEN': MIN_CELIC_GEN, 'C_LOGREG': C_LOGREG,
               'N_BOOT': N_BOOT, 'SEME': SEME, 'humangem': HUMANGEM_URL, 'zagonov_scfea': len(fluksi),
               'geni': {k: int(len(v)) for k, v in nabori.items()},
               'verzije': {'pandas': pd.__version__, 'numpy': np.__version__}}, f, indent=2, ensure_ascii=False)
print('Shranjeno v', OUT_DIR)
for ime in sorted(os.listdir(OUT_DIR)):
    print(' ', ime)